# Customer Analytics Group Project - Technical Report
# Credit card default risk segmentation

**Group Members:** Mayer Attie, Ian Fernandez, Jose Bouza, Ivan Ryazantsev

**Date:** 6/04/2026

**Dataset:** Default of Credit Card Clients (UCI ML Repository, ID 350), 30,000 Taiwanese credit card holders

---


## 1. Executive summary

Not all credit card clients carry the same risk, so they should not all be managed the same way. Using data from 30,000 Taiwanese credit card holders (22.1% of whom failed to make their next payment), we segmented clients into four risk groups and explored what drives default.

The segmentation produced groups with meaningfully different default rates: from 12.7% (low risk) to 60.5% (high risk), a nearly 5x gap. We trained four predictive models to assign each client a continuous risk score (0-100). The best performer, a Random Forest, reached an AUC of 0.770, with a top-decile lift of 3.07x, meaning the riskiest 10% of clients identified by the model default at 3x the portfolio average.

The strongest predictors are all behavioral: how often a client has been late, how severe those delays were, and how much of their credit limit they use. Demographic features like age, gender, and education are weak predictors on their own (AUC = 0.605 when used alone, barely above random). This means the real value is in monitoring payment behavior over time, not in screening at application. The first missed payment is the single most actionable signal: it nearly triples the default rate from 11.7% to 29.8%.


## 2. Problem definition and business context

### 2.1 Business problem

Credit card default is a major source of loss for banks. In this dataset, 22.1% of 30,000 clients failed to make their next monthly payment. But that headline number masks wide variation: some behavioral profiles default at 13%, others at over 60%.

This variation creates an opportunity. If the bank can identify distinct risk groups within its portfolio, it can allocate resources more efficiently: focusing intervention on clients most likely to default, and rewarding reliable clients to retain them.

### 2.2 Research questions

1. Can we identify distinct risk segments with meaningfully different default rates and behavioral profiles?
2. What behavioral and demographic patterns characterize each segment?
3. What early signals indicate a client is moving toward delinquency, and when is the best moment to intervene?

### 2.3 Success metrics

- **Segmentation quality:** at least 3 segments with clearly different default rates
- **Segment interpretability:** profiles that a non-technical stakeholder can understand and act on
- **Predictive support:** a risk scoring model with AUC > 0.75 and top-decile lift > 2x
- **Actionability:** specific recommendations for how to treat each segment differently (Section 8)


## 3. Data overview and preparation

### 3.1 Dataset description

- **Source:** UCI Machine Learning Repository (ID 350), "Default of Credit Card Clients"
- **Size:** 30,000 observations, 23 features + 1 binary target
- **Time period:** April to September 2005 (6 months of payment history)
- **Geography:** Taiwan
- **Key variables:**
  - Demographics: sex, education, marital status, age
  - Credit: LIMIT_BAL (credit limit in NT$)
  - Payment history: PAY_0 to PAY_6 (repayment status per month; -1 = paid on time, 1+ = months delayed)
  - Bill amounts: BILL_AMT1 to BILL_AMT6 (monthly statements)
  - Payment amounts: PAY_AMT1 to PAY_AMT6 (monthly payments made)
  - Target: "default payment next month" (1 = failed to pay, 0 = paid). This tracks whether the client missed their next individual monthly payment, not whether they were formally declared in default by the bank.


In [ ]:
# ============================================================
# Install and import all required libraries
# ============================================================
import subprocess, sys
def install(pkg):
    try: __import__(pkg.split('[')[0].replace('-','_'))
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['ucimlrepo', 'xgboost', 'shap', 'imbalanced-learn']:
    install(pkg)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.cluster import KMeans
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, auc, silhouette_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print('All libraries loaded successfully.')


In [ ]:
# ============================================================
# Load dataset from UCI repository
# ============================================================
import ssl
ssl._create_default_https_context = ssl._create_unverified_context  # fix for local SSL issues

from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=350)
X = dataset.data.features
y = dataset.data.targets

# Combine into single DataFrame
df = pd.concat([X, y], axis=1)

# Rename columns using descriptive names from the dataset metadata
rename_map = {
    'X1': 'LIMIT_BAL', 'X2': 'SEX', 'X3': 'EDUCATION', 'X4': 'MARRIAGE', 'X5': 'AGE',
    'X6': 'PAY_0', 'X7': 'PAY_2', 'X8': 'PAY_3', 'X9': 'PAY_4', 'X10': 'PAY_5', 'X11': 'PAY_6',
    'X12': 'BILL_AMT1', 'X13': 'BILL_AMT2', 'X14': 'BILL_AMT3',
    'X15': 'BILL_AMT4', 'X16': 'BILL_AMT5', 'X17': 'BILL_AMT6',
    'X18': 'PAY_AMT1', 'X19': 'PAY_AMT2', 'X20': 'PAY_AMT3',
    'X21': 'PAY_AMT4', 'X22': 'PAY_AMT5', 'X23': 'PAY_AMT6',
    'Y': 'DEFAULT'
}
df = df.rename(columns=rename_map)

print(f'Dataset shape: {df.shape}')
print(f'Default rate: {df["DEFAULT"].mean():.1%}')
df.head()


In [ ]:
# ============================================================
# 3.2 Data Exploration and Quality Assessment
# ============================================================
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\n" + "=" * 60)
print("STATISTICAL SUMMARY")
print("=" * 60)
df.describe().round(1)

**What we found:**

- No missing values across all 30,000 records.
- 35 duplicate rows, which we remove during cleaning.
- Several undocumented values in the categorical variables:
  - EDUCATION has values 0, 5, and 6 (14, 280, and 51 records respectively) beyond the defined 1-4 scale. We recode these to 4 ("Others").
  - MARRIAGE has 54 records with value 0, which we recode to 3 ("Others").
  - PAY_0 through PAY_6 contain values -2 and 0 that the documentation does not explicitly define. Value -2 appears to mean "no consumption that month" (2,759 records for PAY_0), and 0 means "revolving credit / minimum payment made" (14,737 records for PAY_0, the most common status). We keep these as-is since they carry distinct behavioral meaning.
- Credit limits range from NT$10,000 to NT$1,000,000 with a median of NT$140,000 and mean of NT$167,484, a right-skewed distribution typical of credit portfolios.


In [ ]:
# ============================================================
# 3.3 Data Cleaning and Feature Engineering
# ============================================================

# --- Data Cleaning ---
# Remove duplicate rows
n_before = len(df)
df = df.drop_duplicates()
print(f"Removed {n_before - len(df)} duplicate rows. Remaining: {len(df):,}")

# Recode undocumented EDUCATION values (0, 5, 6) to 4 ("Others")
df['EDUCATION'] = df['EDUCATION'].replace([0, 5, 6], 4)

# Recode undocumented MARRIAGE value (0) to 3 ("Others")
df['MARRIAGE'] = df['MARRIAGE'].replace(0, 3)

# Create readable label mappings for later use
edu_labels = {1: 'Graduate School', 2: 'University', 3: 'High School', 4: 'Others'}
mar_labels = {1: 'Married', 2: 'Single', 3: 'Others'}
sex_labels = {1: 'Male', 2: 'Female'}

print("After cleaning:")
print(f"EDUCATION unique values: {sorted(df['EDUCATION'].unique())}")
print(f"MARRIAGE unique values: {sorted(df['MARRIAGE'].unique())}")

# --- Feature Engineering ---
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
bill_cols = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
pay_amt_cols = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

# 1. Credit Utilization Ratio (per month): bill / credit limit
for i, col in enumerate(bill_cols, 1):
    df[f'UTIL_RATIO_{i}'] = df[col] / df['LIMIT_BAL']

# Average utilization across 6 months
util_cols = [f'UTIL_RATIO_{i}' for i in range(1, 7)]
df['AVG_UTIL'] = df[util_cols].mean(axis=1)

# 2. Payment-to-Bill Ratio (per month): amount paid / bill amount
for i, (pay_col, bill_col) in enumerate(zip(pay_amt_cols, bill_cols), 1):
    # Avoid division by zero: if bill is 0 or negative, ratio = 1 (fully paid)
    df[f'PAY_RATIO_{i}'] = np.where(df[bill_col] > 0, df[pay_col] / df[bill_col], 1.0)
    df[f'PAY_RATIO_{i}'] = df[f'PAY_RATIO_{i}'].clip(0, 5)  # Cap extreme values

# Average payment ratio
pay_ratio_cols = [f'PAY_RATIO_{i}' for i in range(1, 7)]
df['AVG_PAY_RATIO'] = df[pay_ratio_cols].mean(axis=1)

# 3. Payment Delay Features
df['MAX_DELAY'] = df[pay_cols].max(axis=1)     # Worst delinquency in 6 months
df['AVG_DELAY'] = df[pay_cols].mean(axis=1)    # Average delinquency
df['NUM_LATE'] = (df[pay_cols] > 0).sum(axis=1)  # Number of months with late payment

# 4. Bill Growth (trend): is debt increasing or decreasing?
# Positive = debt growing, negative = debt shrinking
df['BILL_GROWTH'] = (df['BILL_AMT1'] - df['BILL_AMT6']) / (df['LIMIT_BAL'] + 1)

# 5. Payment Consistency (std of payment amounts — lower = more consistent)
df['PAY_CONSISTENCY'] = df[pay_amt_cols].std(axis=1)

# 6. Age Bins for demographic analysis
df['AGE_GROUP'] = pd.cut(df['AGE'], bins=[20, 30, 40, 50, 60, 80],
                         labels=['21-30', '31-40', '41-50', '51-60', '61+'])

print(f"\nFeature-engineered dataset shape: {df.shape}")
print(f"New features added: {df.shape[1] - 24}")
df[['LIMIT_BAL', 'AVG_UTIL', 'AVG_PAY_RATIO', 'MAX_DELAY', 'NUM_LATE', 'BILL_GROWTH', 'DEFAULT']].describe().round(3)

**Why these cleaning and engineering decisions:**

**Cleaning:**
- We removed 35 duplicate rows to avoid biasing model training.
- EDUCATION and MARRIAGE recoding groups small, ambiguous categories together without losing observations.
- PAY_0 through PAY_6 values of -2 and 0 are retained because they represent distinct behaviors (no activity vs. revolving credit). Collapsing them would lose information.

**Feature engineering:**
- **Credit utilization (AVG_UTIL):** a standard risk indicator in credit scoring. Clients using a high share of their credit limit tend to default more often.
- **Payment ratio (AVG_PAY_RATIO):** captures how much of their bill clients actually pay. Low ratios mean minimum-payment behavior, an early warning sign.
- **Delay features (MAX_DELAY, NUM_LATE):** aggregate payment delays across the full 6-month window. As we will see, these turn out to be the strongest predictors, outperforming any single month's status.
- **Bill growth:** tracks whether debt is increasing or decreasing relative to the credit limit.
- **Payment consistency:** high variability in payment amounts may reflect irregular income.


In [ ]:
# ============================================================
# 4.1 Target Variable and Demographic Distributions
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Target distribution
default_counts = df['DEFAULT'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0, 0].bar(['No Default (0)', 'Default (1)'], default_counts.values, color=colors, edgecolor='black')
for i, v in enumerate(default_counts.values):
    axes[0, 0].text(i, v + 200, f'{v:,}\n({v/len(df):.1%})', ha='center', fontweight='bold')
axes[0, 0].set_title('Target Variable: Default Status', fontsize=13, fontweight='bold')
axes[0, 0].set_ylabel('Count')

# Default rate by Gender
sex_default = df.groupby('SEX')['DEFAULT'].mean()
bars = axes[0, 1].bar([sex_labels[k] for k in sex_default.index], sex_default.values * 100,
                       color=['#3498db', '#e91e63'], edgecolor='black')
for bar, val in zip(bars, sex_default.values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{val:.1%}', ha='center', fontweight='bold')
axes[0, 1].set_title('Default Rate by Gender', fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Default Rate (%)')

# Default rate by Education
edu_default = df.groupby('EDUCATION')['DEFAULT'].mean().sort_index()
bars = axes[0, 2].bar([edu_labels[k] for k in edu_default.index], edu_default.values * 100,
                       color='#9b59b6', edgecolor='black')
for bar, val in zip(bars, edu_default.values):
    axes[0, 2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{val:.1%}', ha='center', fontweight='bold', fontsize=9)
axes[0, 2].set_title('Default Rate by Education', fontsize=13, fontweight='bold')
axes[0, 2].set_ylabel('Default Rate (%)')
axes[0, 2].tick_params(axis='x', rotation=15)

# Default rate by Marital Status
mar_default = df.groupby('MARRIAGE')['DEFAULT'].mean().sort_index()
bars = axes[1, 0].bar([mar_labels[k] for k in mar_default.index], mar_default.values * 100,
                       color='#e67e22', edgecolor='black')
for bar, val in zip(bars, mar_default.values):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{val:.1%}', ha='center', fontweight='bold')
axes[1, 0].set_title('Default Rate by Marital Status', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Default Rate (%)')

# Default rate by Age Group
age_default = df.groupby('AGE_GROUP', observed=True)['DEFAULT'].mean()
bars = axes[1, 1].bar(age_default.index.astype(str), age_default.values * 100,
                       color='#1abc9c', edgecolor='black')
for bar, val in zip(bars, age_default.values):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{val:.1%}', ha='center', fontweight='bold')
axes[1, 1].set_title('Default Rate by Age Group', fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel('Default Rate (%)')

# Credit Limit distribution by default status
for default_val, color, label in [(0, '#2ecc71', 'No Default'), (1, '#e74c3c', 'Default')]:
    axes[1, 2].hist(df[df['DEFAULT'] == default_val]['LIMIT_BAL'] / 1000,
                    bins=30, alpha=0.6, color=color, label=label, edgecolor='black')
axes[1, 2].set_title('Credit Limit Distribution by Default', fontsize=13, fontweight='bold')
axes[1, 2].set_xlabel('Credit Limit (NT$ thousands)')
axes[1, 2].set_ylabel('Count')
axes[1, 2].legend()

plt.tight_layout()
plt.savefig('fig_01_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

**What stands out:**

- **Class imbalance:** 22.1% defaulted (6,636) vs. 77.9% did not (23,364). We address this with SMOTE during model training.
- **Gender:** males default at 24.2%, females at 20.8%, a 3.4 percentage point gap.
- **Education:** high school graduates default at 25.2%, university at 23.7%, graduate school at 19.2%. The "Others" category has an unusually low 7.1%, likely due to its small size (468 clients) and distinct composition.
- **Age:** older clients default more, not less. The 61+ group has the highest rate (26.8%), followed by 51-60 (25.2%), while 31-40 has the lowest (20.4%). This runs counter to the common assumption that younger clients are riskier, and may reflect accumulated debt or income decline in this portfolio.
- **Marital status:** married clients (23.5%) default slightly more than single ones (20.9%), possibly due to higher household financial obligations.
- **Credit limit:** defaulters hold lower average limits (NT$130,110 vs. NT$178,100 for non-defaulters), consistent with the bank already applying some risk-based limit-setting.
- **Late payments:** the clearest pattern in the data. Clients with zero late months default at 11.7%. One late month jumps to 29.8%. Six late months: 70.3%. Each additional late month roughly doubles the risk.

The near-linear relationship between NUM_LATE and default rate is the most actionable finding from the EDA. A single missed payment is not just a minor blip; it is a 2.5x risk increase from baseline and the clearest trigger point for intervention.


### 4.2 Payment behavior and financial patterns


In [ ]:
# ============================================================
# 4.2 Payment Behavior Patterns
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Payment delay heatmap: average delay by month and default status
delay_data = df.groupby('DEFAULT')[pay_cols].mean()
delay_data.index = ['No Default', 'Default']
delay_data.columns = ['Sept', 'Aug', 'Jul', 'Jun', 'May', 'Apr']
sns.heatmap(delay_data, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0, 0],
            linewidths=0.5, cbar_kws={'label': 'Avg Delay Score'})
axes[0, 0].set_title('Average Payment Delay by Month & Default Status', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('')

# Distribution of MAX_DELAY by default status
for default_val, color, label in [(0, '#2ecc71', 'No Default'), (1, '#e74c3c', 'Default')]:
    subset = df[df['DEFAULT'] == default_val]['MAX_DELAY']
    axes[0, 1].hist(subset, bins=range(-3, 10), alpha=0.6, color=color, label=label,
                    edgecolor='black', density=True)
axes[0, 1].set_title('Distribution of Worst Delinquency (MAX_DELAY)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Maximum Payment Delay (months)')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend()

# Average credit utilization by default status
for default_val, color, label in [(0, '#2ecc71', 'No Default'), (1, '#e74c3c', 'Default')]:
    subset = df[df['DEFAULT'] == default_val]['AVG_UTIL'].clip(-0.5, 3)
    axes[1, 0].hist(subset, bins=50, alpha=0.6, color=color, label=label,
                    edgecolor='black', density=True)
axes[1, 0].set_title('Credit Utilization Distribution by Default', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Average Credit Utilization Ratio')
axes[1, 0].set_ylabel('Density')
axes[1, 0].axvline(x=0.8, color='black', linestyle='--', alpha=0.7, label='80% threshold')
axes[1, 0].legend()

# Default rate by number of late payment months
late_default = df.groupby('NUM_LATE')['DEFAULT'].agg(['mean', 'count'])
bars = axes[1, 1].bar(late_default.index, late_default['mean'].values * 100,
                       color=plt.cm.RdYlGn_r(np.linspace(0.2, 0.9, len(late_default))),
                       edgecolor='black')
for bar, (idx, row) in zip(bars, late_default.iterrows()):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{row["mean"]:.0%}\n(n={row["count"]:,.0f})', ha='center', fontsize=8)
axes[1, 1].set_title('Default Rate by # of Late Payment Months', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Number of Months with Late Payment (out of 6)')
axes[1, 1].set_ylabel('Default Rate (%)')

plt.tight_layout()
plt.savefig('fig_02_payment_behavior.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 4.3 Correlation Analysis
# ============================================================
corr_features = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE'] + pay_cols + \
                ['AVG_UTIL', 'AVG_PAY_RATIO', 'MAX_DELAY', 'NUM_LATE', 'BILL_GROWTH', 'DEFAULT']

corr_matrix = df[corr_features].corr()

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Full correlation heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[0], linewidths=0.5, vmin=-1, vmax=1,
            annot_kws={'size': 7})
axes[0].set_title('Correlation Matrix of Key Features', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='both', labelsize=8)

# Correlation with target (bar chart)
target_corr = corr_matrix['DEFAULT'].drop('DEFAULT').sort_values()
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in target_corr.values]
target_corr.plot(kind='barh', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_title('Correlation with Default (Target)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Pearson Correlation Coefficient')
axes[1].axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()
plt.savefig('fig_03_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

**Correlation analysis results:**

- NUM_LATE (our engineered feature) has the strongest correlation with default at r = 0.40, followed by PAY_0 at r = 0.32, and then PAY_2 through PAY_6 in decreasing order. Aggregating late payments across the full 6-month window produces a stronger signal than any single month.
- There is a clear recency effect: more recent months correlate more strongly with default (PAY_0: r = 0.32 vs. PAY_6: r = 0.19).
- LIMIT_BAL is the strongest negatively correlated feature (r = -0.15). Higher limits associate with lower default, reflecting the bank's own prior credit assessment.
- Payment amounts (PAY_AMT1-6) have weak negative correlations (r around -0.05 to -0.07). Bill amounts (BILL_AMT1-6) are near zero. The absolute size of the debt matters less than whether payments are being made.
- All demographic features (SEX, EDUCATION, AGE, MARRIAGE) have correlations below |r| = 0.05. Behavioral features are 5 to 10 times more predictive than demographics.


## 5. Main Analysis

Our analysis has two complementary components:

1. **Risk Segmentation (Section 5.1):** K-Means clustering to divide clients into distinct risk groups, this is the primary deliverable that drives the business strategies in Sections 7-8.
2. **Predictive Modeling (Section 5.2):** Machine learning models to generate individual risk scores (0-100) that support the segmentation by enabling continuous monitoring and resource prioritization within each segment.
3. **Interpretable Rules (Section 5.3):** A decision tree that extracts simple, human-readable risk rules deployable without ML infrastructure.

### 5.1 Risk Segmentation via K-Means Clustering

#### 5.1.1 Methodology

**Objective:** Segment the credit card clients into distinct risk groups based on their financial behavior, enabling differentiated credit management and marketing strategies.

**Method:** K-Means clustering on standardized behavioral features.

**Rationale:** K-Means is well-suited for this task because: (1) it scales well to 30,000 records, (2) it produces clearly defined, non-overlapping segments that are easy for business stakeholders to understand and act on, and (3) the resulting cluster centroids provide intuitive "average profiles" for each segment.

**Feature Selection for Clustering:**
We use behavioral features (not demographics) to ensure segments reflect actual financial behavior:
- `LIMIT_BAL`, credit limit (proxy for creditworthiness)
- `AVG_UTIL`, average credit utilization ratio
- `AVG_PAY_RATIO`, average payment-to-bill ratio
- `MAX_DELAY`, worst delinquency in 6 months
- `NUM_LATE`, number of months with late payment
- `AVG_DELAY`, average delinquency level

**Choosing k=4:** The silhouette analysis shows k=3 achieves the highest silhouette score (0.345), while k=4 scores slightly lower (0.308). However, we select **k=4** because: (1) it separates the extreme delinquent group (60.5% default rate) from the moderate-risk group, which is critical for targeted intervention strategies; (2) the elbow plot shows a clear diminishing-returns bend at k=4; and (3) four segments map naturally to the bank's operational needs (low/low-mod/mod-high/high risk tiers).

**Assumptions:**
1. Behavioral features are more actionable for segmentation than demographics.
2. Euclidean distance on standardized features is an appropriate similarity measure.
3. The marginal loss in silhouette score from k=3 to k=4 is justified by the operational value of a fourth, highly differentiated segment.



In [ ]:
# ============================================================
# 5.1 K-Means Clustering: Optimal K Selection
# ============================================================

# Select behavioral features for clustering
cluster_features = ['LIMIT_BAL', 'AVG_UTIL', 'AVG_PAY_RATIO', 'MAX_DELAY', 'NUM_LATE', 'AVG_DELAY']
X_cluster = df[cluster_features].copy()

# Standardize features
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

# Determine optimal k using Elbow Method + Silhouette Score
K_range = range(2, 9)
inertias = []
silhouette_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(X_cluster_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_cluster_scaled, labels, sample_size=5000, random_state=42))
    print(f"k={k}: Inertia={km.inertia_:,.0f}, Silhouette={silhouette_scores[-1]:.4f}")

# Plot Elbow and Silhouette
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k=4 (chosen)')
axes[0].set_title('Elbow Method: Inertia vs. Number of Clusters', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)')
axes[0].legend()

axes[1].plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
axes[1].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k=4 (chosen)')
axes[1].set_title('Silhouette Score vs. Number of Clusters', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig_04_elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.1 Apply K-Means with k=4 and Profile Segments
# ============================================================
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10, max_iter=300)
df['SEGMENT'] = kmeans.fit_predict(X_cluster_scaled)

# Calculate segment profiles
segment_profile = df.groupby('SEGMENT').agg(
    count=('DEFAULT', 'size'),
    default_rate=('DEFAULT', 'mean'),
    avg_limit=('LIMIT_BAL', 'mean'),
    avg_age=('AGE', 'mean'),
    avg_util=('AVG_UTIL', 'mean'),
    avg_pay_ratio=('AVG_PAY_RATIO', 'mean'),
    avg_max_delay=('MAX_DELAY', 'mean'),
    avg_num_late=('NUM_LATE', 'mean'),
    avg_delay=('AVG_DELAY', 'mean'),
    pct_female=('SEX', lambda x: (x == 2).mean()),
    avg_bill=('BILL_AMT1', 'mean')
).round(3)

segment_profile['pct_of_total'] = (segment_profile['count'] / len(df) * 100).round(1)

# Sort segments by default rate to assign meaningful names
segment_profile = segment_profile.sort_values('default_rate')
segment_order = segment_profile.index.tolist()

# Assign descriptive names based on risk level
segment_names = {}
for i, seg_id in enumerate(segment_order):
    if i == 0:
        segment_names[seg_id] = 'Low Risk - Reliable Payers'
    elif i == 1:
        segment_names[seg_id] = 'Low-Moderate Risk - Careful Users'
    elif i == 2:
        segment_names[seg_id] = 'Moderate-High Risk - Revolvers'
    else:
        segment_names[seg_id] = 'High Risk - Delinquent'

df['SEGMENT_NAME'] = df['SEGMENT'].map(segment_names)
segment_profile['segment_name'] = segment_profile.index.map(segment_names)

# Define ordered names for consistent plotting
ordered_names = ['Low Risk - Reliable Payers', 'Low-Moderate Risk - Careful Users',
                 'Moderate-High Risk - Revolvers', 'High Risk - Delinquent']

print("=" * 80)
print("SEGMENT PROFILES")
print("=" * 80)
for _, row in segment_profile.iterrows():
    print(f"\n--- {row['segment_name']} ---")
    print(f"  Size: {row['count']:,.0f} clients ({row['pct_of_total']}% of total)")
    print(f"  Default Rate: {row['default_rate']:.1%}")
    print(f"  Avg Credit Limit: NT${row['avg_limit']:,.0f}")
    print(f"  Avg Age: {row['avg_age']:.1f}")
    print(f"  Avg Credit Utilization: {row['avg_util']:.2f}")
    print(f"  Avg Payment Ratio: {row['avg_pay_ratio']:.2f}")
    print(f"  Avg Max Delay: {row['avg_max_delay']:.1f} months")
    print(f"  Avg # Late Months: {row['avg_num_late']:.1f}")
    print(f"  % Female: {row['pct_female']:.1%}")

In [ ]:
# ============================================================
# 5.1 Segment Visualization
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

segment_colors = {'Low Risk - Reliable Payers': '#2ecc71',
                  'Low-Moderate Risk - Careful Users': '#3498db',
                  'Moderate-High Risk - Revolvers': '#f39c12',
                  'High Risk - Delinquent': '#e74c3c'}

ordered_names = ['Low Risk - Reliable Payers', 'Low-Moderate Risk - Careful Users',
                 'Moderate-High Risk - Revolvers', 'High Risk - Delinquent']

# Segment sizes (pie chart)
seg_counts = df['SEGMENT_NAME'].value_counts().reindex(ordered_names)
axes[0, 0].pie(seg_counts.values, labels=[f'{name}\n({count:,})' for name, count in zip(seg_counts.index, seg_counts.values)],
               colors=[segment_colors[n] for n in seg_counts.index],
               autopct='%1.1f%%', startangle=90, textprops={'fontsize': 9})
axes[0, 0].set_title('Segment Distribution', fontsize=13, fontweight='bold')

# Default rate by segment
seg_default = df.groupby('SEGMENT_NAME')['DEFAULT'].mean().reindex(ordered_names)
bars = axes[0, 1].bar(range(len(seg_default)), seg_default.values * 100,
                       color=[segment_colors[n] for n in seg_default.index], edgecolor='black')
for bar, val in zip(bars, seg_default.values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.1%}', ha='center', fontweight='bold')
axes[0, 1].set_xticks(range(len(seg_default)))
axes[0, 1].set_xticklabels(['Low Risk', 'Low-Mod', 'Mod-High', 'High Risk'], fontsize=10)
axes[0, 1].set_title('Default Rate by Segment', fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('Default Rate (%)')
axes[0, 1].axhline(y=df['DEFAULT'].mean()*100, color='black', linestyle='--', alpha=0.5, label=f'Overall: {df["DEFAULT"].mean():.1%}')
axes[0, 1].legend()

# Radar chart data: segment profiles on key metrics
seg_means = df.groupby('SEGMENT_NAME')[cluster_features].mean().reindex(ordered_names)
seg_means_scaled = (seg_means - seg_means.min()) / (seg_means.max() - seg_means.min() + 1e-10)

categories = ['Credit Limit', 'Utilization', 'Pay Ratio', 'Max Delay', '# Late', 'Avg Delay']
for i, (seg_name, row) in enumerate(seg_means_scaled.iterrows()):
    axes[1, 0].plot(range(len(categories)), row.values, 'o-', color=segment_colors[seg_name],
                    label=seg_name.split(' - ')[0], linewidth=2, markersize=6)
axes[1, 0].set_xticks(range(len(categories)))
axes[1, 0].set_xticklabels(categories, fontsize=9)
axes[1, 0].set_title('Segment Profiles (Normalized)', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Normalized Value (0-1)')
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(True, alpha=0.3)

# Average credit limit by segment
seg_limit = df.groupby('SEGMENT_NAME')['LIMIT_BAL'].mean().reindex(ordered_names) / 1000
bars = axes[1, 1].bar(range(len(seg_limit)), seg_limit.values,
                       color=[segment_colors[n] for n in seg_limit.index], edgecolor='black')
for bar, val in zip(bars, seg_limit.values):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                    f'NT${val:,.0f}K', ha='center', fontweight='bold', fontsize=9)
axes[1, 1].set_xticks(range(len(seg_limit)))
axes[1, 1].set_xticklabels(['Low Risk', 'Low-Mod', 'Mod-High', 'High Risk'], fontsize=10)
axes[1, 1].set_title('Avg Credit Limit by Segment (NT$ thousands)', fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel('Credit Limit (NT$ K)')

plt.tight_layout()
plt.savefig('fig_05_segments.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Segment summary table
summary_data = []
for seg_name in ordered_names:
    seg_data = df[df['SEGMENT_NAME'] == seg_name]
    summary_data.append({
        'Segment': seg_name,
        'Size': f"{len(seg_data):,} ({len(seg_data)/len(df):.1%})",
        'Default Rate': f"{seg_data['DEFAULT'].mean():.1%}",
        'Avg Limit (NT$K)': f"{seg_data['LIMIT_BAL'].mean()/1000:.0f}",
        'Avg Utilization': f"{seg_data['AVG_UTIL'].mean():.2f}",
        'Avg Max Delay': f"{seg_data['MAX_DELAY'].mean():.1f}",
        'Avg # Late': f"{seg_data['NUM_LATE'].mean():.1f}",
        'Avg Age': f"{seg_data['AGE'].mean():.0f}"
    })
pd.DataFrame(summary_data)

### 5.2 Default prediction models

#### 5.2.1 Methodology

The segmentation tells us which groups exist. The predictive model adds a second layer: a continuous risk score (0-100) for each individual client, so the bank can prioritize resources within and across segments and track how clients move over time.

We compare four models: Logistic Regression (interpretable baseline), Random Forest (ensemble method), XGBoost (gradient boosting), and a Neural Network (MLP). All are trained on the same 80/20 stratified split with SMOTE applied only to the training set.

**A note on the target variable:** "default" in this dataset means the client failed to make their next individual monthly payment, not that they were formally declared in default. A client already several months behind but who makes some payment in the target month is coded as "no default." We are predicting next-month payment behavior, which is narrow but operationally useful.

**Evaluation:** AUC-ROC, lift chart (top decile), precision-recall curves, confusion matrix. We use XGBoost's SHAP values for feature-level interpretability.


In [ ]:
# ============================================================
# 5.2 Prepare Data for Modeling
# ============================================================

# Select features for modeling (exclude target, segment labels, age group, individual util/pay ratios)
exclude_cols = ['DEFAULT', 'SEGMENT', 'SEGMENT_NAME', 'AGE_GROUP'] + \
               [f'UTIL_RATIO_{i}' for i in range(1, 7)] + \
               [f'PAY_RATIO_{i}' for i in range(1, 7)]

feature_cols = [col for col in df.columns if col not in exclude_cols]
X_model = df[feature_cols].copy()
y_model = df['DEFAULT'].copy()

print(f"Features for modeling: {len(feature_cols)}")
print(f"Feature list: {feature_cols}")

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model, test_size=0.2, random_state=42, stratify=y_model)

print(f"\nTrain set: {X_train.shape[0]:,} samples (Default rate: {y_train.mean():.1%})")
print(f"Test set: {X_test.shape[0]:,} samples (Default rate: {y_test.mean():.1%})")

# Apply SMOTE to training data only
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f"\nAfter SMOTE - Train set: {X_train_smote.shape[0]:,} samples (Default rate: {y_train_smote.mean():.1%})")

# Scale features for Logistic Regression and Neural Network
scaler_model = StandardScaler()
X_train_scaled = scaler_model.fit_transform(X_train_smote)
X_test_scaled = scaler_model.transform(X_test)

In [ ]:
# ============================================================
# 5.2 Train All Four Models
# ============================================================
models = {}

# 1. Logistic Regression
print("Training Logistic Regression...")
lr = LogisticRegression(max_iter=1000, random_state=42, C=0.1)
lr.fit(X_train_scaled, y_train_smote)
models['Logistic Regression'] = {'model': lr, 'proba': lr.predict_proba(X_test_scaled)[:, 1]}

# 2. Random Forest
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=20,
                            random_state=42, n_jobs=-1)
rf.fit(X_train_smote, y_train_smote)
models['Random Forest'] = {'model': rf, 'proba': rf.predict_proba(X_test)[:, 1]}

# 3. XGBoost
print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                subsample=0.8, colsample_bytree=0.8,
                                random_state=42, eval_metric='auc',
                                use_label_encoder=False)
xgb_model.fit(X_train_smote, y_train_smote)
models['XGBoost'] = {'model': xgb_model, 'proba': xgb_model.predict_proba(X_test)[:, 1]}

# 4. Neural Network (MLP)
print("Training Neural Network (MLP)...")
mlp = MLPClassifier(hidden_layer_sizes=(64, 32, 16), activation='relu',
                    max_iter=500, random_state=42, early_stopping=True,
                    validation_fraction=0.1, learning_rate='adaptive')
mlp.fit(X_train_scaled, y_train_smote)
models['Neural Network'] = {'model': mlp, 'proba': mlp.predict_proba(X_test_scaled)[:, 1]}

# Calculate AUC for each model
print("\n" + "=" * 60)
print("MODEL PERFORMANCE (AUC-ROC on Test Set)")
print("=" * 60)
for name, data in models.items():
    auc_score = roc_auc_score(y_test, data['proba'])
    data['auc'] = auc_score
    print(f"{name:25s}: AUC = {auc_score:.4f}")

In [ ]:
# ============================================================
# 5.2 Model Evaluation: ROC Curves, Lift Chart, Confusion Matrices
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

model_colors = {'Logistic Regression': '#3498db', 'Random Forest': '#2ecc71',
                'XGBoost': '#e74c3c', 'Neural Network': '#9b59b6'}

# --- ROC Curves ---
for name, data in models.items():
    fpr, tpr, _ = roc_curve(y_test, data['proba'])
    axes[0, 0].plot(fpr, tpr, color=model_colors[name], linewidth=2,
                    label=f"{name} (AUC={data['auc']:.3f})")
axes[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random (AUC=0.500)')
axes[0, 0].set_title('ROC Curves - All Models', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('False Positive Rate')
axes[0, 0].set_ylabel('True Positive Rate')
axes[0, 0].legend(fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# --- Precision-Recall Curves ---
for name, data in models.items():
    precision, recall, _ = precision_recall_curve(y_test, data['proba'])
    pr_auc = auc(recall, precision)
    axes[0, 1].plot(recall, precision, color=model_colors[name], linewidth=2,
                    label=f"{name} (PR-AUC={pr_auc:.3f})")
axes[0, 1].axhline(y=y_test.mean(), color='black', linestyle='--', alpha=0.5, label=f'Baseline ({y_test.mean():.2f})')
axes[0, 1].set_title('Precision-Recall Curves', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Recall')
axes[0, 1].set_ylabel('Precision')
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# --- Lift Chart (using best model) ---
best_name = max(models, key=lambda x: models[x]['auc'])
best_proba = models[best_name]['proba']
lift_df = pd.DataFrame({'actual': y_test.values, 'predicted_proba': best_proba})
lift_df = lift_df.sort_values('predicted_proba', ascending=False)
lift_df['decile'] = pd.qcut(range(len(lift_df)), 10, labels=False) + 1

decile_stats = lift_df.groupby('decile').agg(
    count=('actual', 'size'), defaults=('actual', 'sum'), default_rate=('actual', 'mean'))
overall_default_rate = y_test.mean()
decile_stats['lift'] = decile_stats['default_rate'] / overall_default_rate
decile_stats['cumulative_defaults'] = decile_stats['defaults'].cumsum()
decile_stats['cumulative_pct_defaults'] = decile_stats['cumulative_defaults'] / decile_stats['defaults'].sum()

bars = axes[1, 0].bar(decile_stats.index, decile_stats['lift'].values,
                       color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 10)), edgecolor='black')
axes[1, 0].axhline(y=1.0, color='black', linestyle='--', alpha=0.7, label='Baseline (Lift=1)')
for bar, val in zip(bars, decile_stats['lift'].values):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                    f'{val:.2f}', ha='center', fontsize=9, fontweight='bold')
axes[1, 0].set_title(f'Lift Chart by Decile ({best_name})', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Decile (1 = Highest Risk)')
axes[1, 0].set_ylabel('Lift (vs. Random)')
axes[1, 0].legend()

# --- Cumulative Gains Chart ---
for name, data in models.items():
    sorted_actual = y_test.values[np.argsort(data['proba'])[::-1]]
    cum_defaults = np.cumsum(sorted_actual)
    cum_pct = cum_defaults / sorted_actual.sum()
    pct_population = np.arange(1, len(cum_pct) + 1) / len(cum_pct)
    axes[1, 1].plot(pct_population, cum_pct, color=model_colors[name], linewidth=2, label=name)
axes[1, 1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[1, 1].set_title('Cumulative Gains Chart', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('% of Population (sorted by predicted risk)')
axes[1, 1].set_ylabel('% of Defaults Captured')
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_06_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest model: {best_name} (AUC={models[best_name]['auc']:.4f})")
print(f"\nLift by Decile ({best_name}):")
print(decile_stats[['count', 'defaults', 'default_rate', 'lift', 'cumulative_pct_defaults']].round(3))

In [ ]:
# ============================================================
# 5.2 Feature Importance (XGBoost) and SHAP Analysis
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# XGBoost feature importance (gain)
importance = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
top_15 = importance.tail(15)
top_15.plot(kind='barh', ax=axes[0], color='#e74c3c', edgecolor='black')
axes[0].set_title('Top 15 Features — XGBoost Importance (Gain)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Feature Importance')

# SHAP values for XGBoost (using a sample for speed)
explainer = shap.TreeExplainer(xgb_model)
X_test_sample = X_test.sample(n=min(1000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(X_test_sample)

plt.sca(axes[1])
shap.summary_plot(shap_values, X_test_sample, plot_type='bar', max_display=15,
                  show=False, color='#3498db')
axes[1].set_title('Top 15 Features — SHAP Importance', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('fig_07_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.2 SHAP Beeswarm + Confusion Matrix + Risk Scores
# ============================================================

# SHAP beeswarm plot
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, max_display=15, show=False)
plt.title('SHAP Beeswarm - Feature Impact on Default Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_08_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

# Confusion matrix for best model at optimal threshold
best_name = max(models, key=lambda x: models[x]['auc'])
best_proba = models[best_name]['proba']
from sklearn.metrics import f1_score
thresholds = np.arange(0.2, 0.6, 0.01)
f1_scores = [f1_score(y_test, (best_proba >= t).astype(int)) for t in thresholds]
optimal_threshold = thresholds[np.argmax(f1_scores)]
y_pred_optimal = (best_proba >= optimal_threshold).astype(int)

print(f"Optimal threshold (max F1): {optimal_threshold:.2f}")
print(f"\nClassification Report ({best_name}, threshold={optimal_threshold:.2f}):")
print(classification_report(y_test, y_pred_optimal, target_names=['No Default', 'Default']))

fig, ax = plt.subplots(figsize=(7, 6))
cm = confusion_matrix(y_test, y_pred_optimal)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
            xticklabels=['No Default', 'Default'], yticklabels=['No Default', 'Default'])
ax.set_title(f'Confusion Matrix - {best_name} (threshold={optimal_threshold:.2f})', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('fig_09_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.2 Generate Risk Scores (0-100 scale) for All Clients
# ============================================================
# Use XGBoost to score full dataset (close to best model, supports SHAP)
X_all = df[feature_cols]
all_proba = xgb_model.predict_proba(X_all)[:, 1]
df['RISK_SCORE'] = (all_proba * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for seg_name in ordered_names:
    subset = df[df['SEGMENT_NAME'] == seg_name]['RISK_SCORE']
    axes[0].hist(subset, bins=50, alpha=0.5, label=seg_name.split(' - ')[0],
                 color=segment_colors[seg_name], edgecolor='black', density=True)
axes[0].set_title('Risk Score Distribution by Segment', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Risk Score (0-100)')
axes[0].set_ylabel('Density')
axes[0].legend(fontsize=9)

seg_order = [df[df['SEGMENT_NAME'] == n]['RISK_SCORE'] for n in ordered_names]
bp = axes[1].boxplot(seg_order, labels=['Low', 'Low-Mod', 'Mod-High', 'High'],
                      patch_artist=True, notch=True)
for patch, name in zip(bp['boxes'], ordered_names):
    patch.set_facecolor(segment_colors[name])
axes[1].set_title('Risk Score Distribution by Segment (Box Plot)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Segment')
axes[1].set_ylabel('Risk Score (0-100)')

plt.tight_layout()
plt.savefig('fig_10_risk_scores.png', dpi=150, bbox_inches='tight')
plt.show()

print("Risk Score Statistics by Segment:")
print(df.groupby('SEGMENT_NAME')['RISK_SCORE'].describe().round(1).reindex(ordered_names))

### 5.3 Decision tree for interpretable risk rules

The Random Forest and XGBoost models are black boxes. To complement them, we train a shallow decision tree (max depth = 4) that produces human-readable rules. The goal is not to beat the full model on accuracy, but to give credit officers a simple set of conditions they can apply without running any ML system.


In [ ]:
# ============================================================
# 5.3 Decision Tree - Interpretable Risk Rules
# ============================================================
dt = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, random_state=42, class_weight='balanced')
dt.fit(X_train[feature_cols], y_train)

dt_proba = dt.predict_proba(X_test[feature_cols])[:, 1]
dt_auc = roc_auc_score(y_test, dt_proba)
print(f"Decision Tree AUC: {dt_auc:.4f} (vs. best model: {models[max(models, key=lambda x: models[x]['auc'])]['auc']:.4f})")
print(f"\nNote: The decision tree sacrifices some accuracy for interpretability.")

fig, ax = plt.subplots(figsize=(24, 12))
plot_tree(dt, feature_names=feature_cols, class_names=['No Default', 'Default'],
          filled=True, rounded=True, ax=ax, fontsize=8, proportion=True, impurity=False)
ax.set_title('Decision Tree for Credit Default Risk (max_depth=4)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_11_decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nDecision Tree Rules:")
print(export_text(dt, feature_names=feature_cols, max_depth=3))

### 5.4 Onboarding model: can we predict default without payment history?

The full model relies heavily on payment behavior, which only exists for clients who have been active for several months. A natural follow-up question: what if we strip away all behavioral features and try to predict default using only what the bank knows at the time of application?

We test this with just 5 features: LIMIT_BAL, SEX, EDUCATION, MARRIAGE, and AGE. These are the only variables available before a client makes their first transaction.


In [ ]:
# ============================================================
# 5.4 Onboarding model: demographics + credit limit only
# ============================================================
onboarding_features = ['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE']

X_onb = df[onboarding_features]
y_onb = df['DEFAULT']

X_onb_train, X_onb_test, y_onb_train, y_onb_test = train_test_split(
    X_onb, y_onb, test_size=0.2, random_state=42, stratify=y_onb)

X_onb_tr_s, y_onb_tr_s = SMOTE(random_state=42).fit_resample(X_onb_train, y_onb_train)

scaler_onb = StandardScaler()
X_onb_tr_sc = scaler_onb.fit_transform(X_onb_tr_s)
X_onb_te_sc = scaler_onb.transform(X_onb_test)

# Train same model types for direct comparison
onb_models = {}

lr_onb = LogisticRegression(max_iter=1000, random_state=42)
lr_onb.fit(X_onb_tr_sc, y_onb_tr_s)
onb_models['Logistic Regression'] = lr_onb.predict_proba(X_onb_te_sc)[:, 1]

rf_onb = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=20,
                                random_state=42, n_jobs=-1)
rf_onb.fit(X_onb_tr_s, y_onb_tr_s)
onb_models['Random Forest'] = rf_onb.predict_proba(X_onb_test)[:, 1]

xgb_onb = xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                              random_state=42, eval_metric='auc')
xgb_onb.fit(X_onb_tr_s, y_onb_tr_s)
onb_models['XGBoost'] = xgb_onb.predict_proba(X_onb_test)[:, 1]

# Compare AUCs
print('Model comparison: onboarding features vs. full behavioral model')
print('=' * 65)
print(f'{"Model":25s} {"Onboarding AUC":>15s} {"Full Model AUC":>15s}')
print('=' * 65)
for name, proba in onb_models.items():
    onb_auc = roc_auc_score(y_onb_test, proba)
    full_auc = models[name]['auc']
    print(f'{name:25s} {onb_auc:>14.4f}  {full_auc:>14.4f}')

# ROC comparison plot
fig, ax = plt.subplots(figsize=(10, 7))
for name, proba in onb_models.items():
    fpr, tpr, _ = roc_curve(y_onb_test, proba)
    onb_auc = roc_auc_score(y_onb_test, proba)
    ax.plot(fpr, tpr, linewidth=2, linestyle='--',
            label=f'{name} - onboarding (AUC={onb_auc:.3f})')

for name, data in models.items():
    if name in ['Logistic Regression', 'Random Forest', 'XGBoost']:
        fpr, tpr, _ = roc_curve(y_test, data['proba'])
        ax.plot(fpr, tpr, linewidth=2,
                label=f'{name} - full model (AUC={data["auc"]:.3f})')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_title('ROC curves: onboarding features (dashed) vs. full model (solid)', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_12_onboarding_vs_full.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Onboarding score quartile analysis
X_all_onb_sc = scaler_onb.transform(df[onboarding_features])
df['ONBOARDING_SCORE'] = (lr_onb.predict_proba(X_all_onb_sc)[:, 1] * 100).round(1)
df['RISK_QUARTILE'] = pd.qcut(df['ONBOARDING_SCORE'], 4,
                               labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'])

fig, ax = plt.subplots(figsize=(9, 6))
q_stats = df.groupby('RISK_QUARTILE', observed=True)['DEFAULT'].agg(['mean', 'count'])
colors_q = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
bars = ax.bar(range(4), q_stats['mean'].values * 100, color=colors_q, edgecolor='black')
for bar, (idx, row) in zip(bars, q_stats.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{row["mean"]:.1%}\n(n={row["count"]:,.0f})', ha='center', fontweight='bold')
ax.set_xticks(range(4))
ax.set_xticklabels(['Q1\n(Lowest Risk)', 'Q2', 'Q3', 'Q4\n(Highest Risk)'])
ax.set_title('Default rate by onboarding risk score quartile', fontsize=13, fontweight='bold')
ax.set_ylabel('Default Rate (%)')
ax.axhline(y=df['DEFAULT'].mean()*100, color='black', linestyle='--', alpha=0.5,
           label=f'Overall: {df["DEFAULT"].mean():.1%}')
ax.legend()
plt.tight_layout()
plt.savefig('fig_13_onboarding_quartiles.png', dpi=150, bbox_inches='tight')
plt.show()

q1 = q_stats.iloc[0]['mean']
q4 = q_stats.iloc[3]['mean']
print(f'Q1 default rate: {q1:.1%}, Q4: {q4:.1%} (ratio: {q4/q1:.1f}x)')
print(f'Behavioral segmentation achieves ~5x (12.7% vs 60.5%)')


**Findings from the onboarding model:**

The best onboarding model (Random Forest) reaches an AUC of 0.605, compared to 0.770 for the full behavioral model. That gap of 0.165 is substantial. With only 5 features, the model barely outperforms random guessing (AUC = 0.50).

Still, the model is not entirely useless. Splitting clients into quartiles by onboarding score produces default rates from about 14% (Q1) to 30% (Q4), a roughly 2x separation. The behavioral segmentation achieves nearly 5x. So onboarding features offer some directional signal, just not enough to make reliable individual decisions.

LIMIT_BAL carries most of the weight, which creates a circularity problem: the bank already used its own risk judgment to set the credit limit, so the model is partly reflecting the bank's prior decision back to it.

The practical takeaway is that a credit risk system needs two layers. An onboarding scorecard (limited but better than nothing) for initial decisions, and a behavioral monitoring model that takes over once the client builds payment history, typically after 3 to 6 months of activity. The performance gap between the two models is itself the strongest argument for investing in behavioral monitoring.


## 6. Key insights and findings

### 6.1 Main findings

**Finding 1: Engineered delay features dominate the prediction**

MAX_DELAY (worst delinquency across 6 months) is the single most important feature in XGBoost, with an importance score of 0.209. AVG_DELAY and NUM_LATE follow. These engineered aggregates outperform any individual month's payment status (PAY_0 importance: 0.042). The correlation analysis confirms this: NUM_LATE has the highest linear correlation with default at r = 0.40.

However, there is a practical tension here. These features measure past delinquency, meaning they are most informative for clients who are already in trouble. The model's real value is not in flagging someone who is 4 months late (that is already obvious) but in catching the transition from 0 to 1 late payments, where default risk jumps from 11.7% to 29.8%. That first missed payment is the highest-leverage intervention point. See Section 9.1 for a deeper discussion of this limitation.

**Finding 2: Payment behavior creates a near-linear risk escalation**

Default rates by number of late months: 0 late = 11.7%, 1 = 29.8%, 2 = 38.8%, 3 = 50.9%, 4 = 57.3%, 5 = 57.4%, 6 = 70.3%. The jump from 0 to 1 late payments is the sharpest increase (2.5x). After that, risk continues to climb but with diminishing increments. This pattern holds across all 29,965 records with large sample sizes at each level, so it is robust.

**Finding 3: Four risk segments with a 5x default rate gap**

K-Means clustering on behavioral features produced four segments:
- Low Risk, Reliable Payers: 5,510 clients (18.4%), 12.7% default rate
- Low-Moderate Risk, Careful Users: 9,375 clients (31.3%), 15.9% default rate
- Moderate-High Risk, Revolvers: 11,379 clients (38.0%), 19.3% default rate
- High Risk, Delinquent: 3,701 clients (12.4%), 60.5% default rate

The High Risk segment is clearly separated from the rest. It accounts for 12.4% of clients but generates a disproportionate share of defaults. The three lower segments are more closely spaced (12.7% to 19.3%), which is reflected in the silhouette score of 0.308. The separation between those three is real but moderate.

**Finding 4: Demographics produce unexpected patterns but weak signals**

Older clients (61+: 26.8%) default more than younger ones (31-40: 20.4%). Males (24.2%) default more than females (20.8%). Married clients (23.5%) default more than single (20.9%). The age pattern contradicts the common assumption that younger borrowers are riskier.

However, all demographic correlations with default are below |r| = 0.05. These patterns exist but are far too weak to drive credit decisions. The onboarding model (Section 5.4), which uses only demographics and credit limit, achieves an AUC of just 0.605, confirming this.

**Finding 5: Random Forest performs best, but all tree models are close**

| Model | AUC |
|---|---|
| Random Forest | 0.770 |
| XGBoost | 0.763 |
| Decision Tree | 0.761 |
| Neural Network | 0.734 |
| Logistic Regression | 0.720 |

The top decile of the Random Forest captures 30.8% of all defaults at a lift of 3.07x. The top two deciles capture 49.2%. The tree-based models cluster together (0.761 to 0.770), suggesting the data has a natural ceiling around that range given the available features.

**Finding 6: Simple decision tree rules nearly match full model accuracy**

The decision tree (AUC = 0.761) splits first on MAX_DELAY <= 0.5, separating clients who have never been seriously late from those who have. For clients with MAX_DELAY > 0.5, the next split is PAY_0 <= 1.5. Clients with at least one serious past delinquency and a current delay of 2+ months are classified as high risk regardless of other features. This two-rule heuristic captures most of the predictive power and can be deployed without any ML infrastructure.

### 6.2 Answers to research questions

**RQ1: Can we identify distinct risk segments?**

Yes. Four segments with default rates from 12.7% to 60.5%. The segments differ not just in default rate but in behavioral profile: the Low Risk segment has high credit limits (avg NT$311K) and low utilization (22%), while the High Risk segment has low limits (avg NT$88K) and 4.5 out of 6 months with late payments on average.

**RQ2: What patterns characterize each segment?**

The segments are primarily defined by payment delay behavior and credit utilization, not demographics. The Low-Moderate Risk segment stands out for very low utilization (4.5%) and a habit of paying early (avg PAY status is -1.2). The Moderate-High Risk segment is the largest (38% of clients) and is characterized by high utilization (64%) combined with minimal payments. Demographics vary only modestly across segments.

**RQ3: What early signals predict delinquency, and when should the bank intervene?**

The strongest signals are MAX_DELAY, AVG_DELAY, and NUM_LATE. The best moment to intervene is at the first missed payment, where risk jumps 2.5x. Pre-delinquency signals (rising utilization, declining payment ratios) can identify clients trending toward trouble before they miss a payment, but these signals are weaker and less reliable on their own. The decision tree rule, "MAX_DELAY > 0 and PAY_0 >= 2 means high risk," captures most of the predictive signal in a form anyone can apply.


## 7. Business recommendations

### 7.1 Strategic recommendations

**Recommendation 1: Deploy monthly risk scoring**

Use the Random Forest model (AUC 0.770) to assign a 0-100 risk score to every client monthly. Integrate scores into the CRM so account managers can see each client's risk at a glance. The top 20% of scores capture about half of all future defaults, making this a practical tool for prioritizing attention.

**Recommendation 2: Differentiate credit limit policies by segment**

The Low Risk segment has an average limit of NT$311K and a 12.7% default rate. The High Risk segment has NT$88K and 60.5%. These groups should not be managed the same way. Consider gradual limit increases for consistently reliable clients, and limit freezes or reductions for clients with rising risk scores. The High Risk segment (3,701 clients, 12.4% of the portfolio) drives a disproportionate share of losses, so capping further exposure there has an outsized effect.

**Recommendation 3: Target intervention at the Moderate-High Risk segment**

The Moderate-High Risk "Revolvers" are the largest segment (11,379 clients, 38% of portfolio) with a 19.3% default rate. They are characterized by high utilization and minimum payments but are not yet deeply delinquent. This makes them the highest-ROI target for intervention: the group is large enough to matter, and their risk is elevated but not yet entrenched. Research published in the Journal of Banking & Finance (2022) found that repayment reminders reduced delinquency progression by about 5% over 6-12 months. Applied to this segment, that would mean roughly 110 fewer defaults.

**Recommendation 4: Implement decision tree alert rules as a quick win**

The decision tree achieves AUC 0.761 using just two splits: MAX_DELAY > 0 and PAY_0 >= 2. This can be implemented as a simple business rule in the bank's existing systems without any ML infrastructure. When a client first triggers this rule, flag the account for immediate follow-up. This costs almost nothing to deploy and captures most of the full model's discriminative power.

**Recommendation 5: Financial awareness outreach for higher-risk demographics**

Older male clients show higher default rates (males: 24.2%, 51+ age group: 25-27%). While demographics are weak predictors, they can inform how outreach is delivered. Payment scheduling tools, utilization alerts, and debt management tips may be particularly useful for these groups.

### 7.2 Prioritization

**Immediate (month 1-2):** Recommendations 1 and 4. Deploy risk scores and the decision tree alert rule. These are the foundation for everything else.

**Short term (month 3-6):** Recommendation 3. Launch targeted outreach to the Moderate-High Risk segment. This has the highest expected return per dollar spent.

**Medium term (month 5-12):** Recommendations 2 and 5. Credit limit policy changes require risk committee approval. Financial literacy programs take time to design and measure.


## 8. Implementation roadmap: how insights translate to action

### 8.1 Overview

The analysis produces three outputs the bank can act on:
1. A risk segmentation of the full portfolio into four groups with different default profiles
2. A monthly risk score (0-100) for each client based on the Random Forest model
3. A simple decision tree rule that flags high risk accounts without ML infrastructure

Below we describe how each segment should be treated differently, what campaigns could look like, and what the expected financial impact is (with assumptions stated).

---

### 8.2 Segment-specific strategies

#### Segment 1: Low Risk, Reliable Payers (5,510 clients, 18.4%, 12.7% default rate)

Profile: high credit limits (avg NT$311K), low utilization (22%), rarely late, slightly older on average (36 years).

**Initiative 1: Retention and loyalty rewards**

These clients are the bank's most reliable. The risk here is not default but attrition: if a competitor offers better terms, these clients have the credit profile to leave. Proactive limit increases (for clients with 12+ months of clean history and risk scores below 15) and reward programs (cashback, points multipliers) give them a reason to stay.

- Channel: email (50%), app notification (30%), direct mail (20%)
- Frequency: annual review with quarterly check-ins
- Example message: "Based on your payment history, your credit limit has been increased to NT$X. You also qualify for 2x reward points on dining and travel."
- Success metrics: retention rate, average card spending, cross-sell conversion rate

**Initiative 2: Cross-selling financial products**

These are the bank's most creditworthy clients. They are natural candidates for pre-approved personal loans, insurance, or investment products at preferential rates.

- Channel: relationship manager calls (40%), email (40%), branch (20%)
- Frequency: quarterly offers

---

#### Segment 2: Low-Moderate Risk, Careful Users (9,375 clients, 31.3%, 15.9% default rate)

Profile: moderate credit limits (avg NT$207K), very low utilization (4.5%), tend to pay early (avg PAY status is -1.2). These clients manage their credit conservatively.

**Initiative 1: Payment streak incentives**

Reward continued on-time behavior to prevent migration into higher-risk segments. A simple program: 6 consecutive on-time payments earns a cashback bonus or interest rate reduction.

- Channel: email (50%), app push (30%), SMS (20%)
- Example message: "You're 2 months away from earning NT$500 cashback. Keep up your payment streak."
- Success metrics: on-time payment rate (target: 85% to 92%), migration rate to higher-risk segments

**Initiative 2: Utilization monitoring alerts**

Proactive notifications when utilization approaches 70%, with tips on payment scheduling. This is a low cost, automated intervention.

- Channel: app push (60%), SMS (40%)
- Trigger: utilization > 70% in any month
- Success metrics: percentage of clients crossing the 80% threshold

---

#### Segment 3: Moderate-High Risk, Revolvers (11,379 clients, 38.0%, 19.3% default rate)

Profile: low credit limits (avg NT$91K), high utilization (64%), minimum payments, occasional late payments. This is the largest segment and the one where intervention has the highest expected return.

**Initiative 1: Pre-delinquency outreach (4-touch sequence)**

When a client in this segment shows a risk score increase of more than 15 points in a single month or misses their first payment, trigger a structured outreach sequence:

- Touch 1 (day 1): email with payment link and customer service number. Tone: helpful, not punitive. "We noticed your recent payment was delayed. Here's a quick link to pay, or call us to discuss options."
- Touch 2 (day 7): SMS with restructuring offer. "We can help adjust your payment schedule. Reply YES to speak with an advisor."
- Touch 3 (day 14): phone call from an account specialist offering a payment plan (3-6 monthly installments at reduced interest), temporary limit reduction, or financial counseling referral.
- Touch 4 (day 21): formal letter with options before account escalation.

Research basis: a 2022 study in the Journal of Banking & Finance found that repayment reminders reduced average delinquency stage by about 5% over 6-12 months. BAI Banking Strategies reports that pre-delinquency management programs, which proactively identify and restructure at-risk accounts, can prevent a portion of early-stage delinquencies from worsening.

Applied to this segment (11,379 clients, 19.3% default rate = ~2,196 expected defaults): a 5-10% reduction would mean 110-220 fewer defaults. At an average outstanding bill of NT$69,023 per Moderate-High Risk defaulter and 75% LGD (Basel standard for unsecured credit cards), that translates to roughly NT$5.7M to NT$11.4M in prevented losses.

**Initiative 2: Limit freezes for high utilization clients**

For clients with utilization above 90%, freeze further limit increases and consider soft spending caps. Notify the client: "We've paused your limit review while you work on reducing your balance. We'll reassess in 6 months."

- Success metrics: average utilization (target: 80% down to 65%), cure rate (clients returning to good standing)

---

#### Segment 4: High Risk, Delinquent (3,701 clients, 12.4%, 60.5% default rate)

Profile: low credit limits (avg NT$88K), 4.5 out of 6 months late on average, high utilization (61%). These clients are already deeply delinquent.

**Initiative 1: Intensive recovery and restructuring**

Assign dedicated specialists. Offer aggressive restructuring: interest rate reductions, balance split into 12 monthly installments, temporary payment pauses for hardship. The goal is to recover what is possible and prevent full charge-off.

- Channel: phone (50%), email (30%), formal letter (20%)
- Frequency: weekly contact until resolution
- Example: "We want to help you get back on track. We can offer: (a) 50% interest rate reduction for 6 months, (b) balance split into 12 payments, (c) 1-month payment pause if you're facing hardship."

**Initiative 2: Credit limit reduction**

Reduce limits to the current outstanding balance to prevent further spending. Place accounts under enhanced monitoring (weekly risk score updates instead of monthly).

- Success metrics: recovery rate, time to resolution, charge-off rate

---

### 8.3 Cross-segment initiatives

#### 8.3.1 Automated risk scoring engine

Score all clients monthly using the Random Forest model. Push scores and segment labels to the CRM. Flag any client whose score rises by more than 20 points in a single month for immediate review.

#### 8.3.2 Tiered alert system

- Risk score 50-74: automated email + app notification with payment reminders
- Risk score 75-89: SMS + phone call from account specialist with restructuring offer
- Risk score 90+: immediate limit freeze + dedicated recovery specialist

#### 8.3.3 Budget allocation by segment

Rather than spending equally, weight resources toward the segments where intervention has the highest expected impact:

| Segment | Suggested share | Rationale |
|---|---|---|
| Low Risk | ~15% | Low default rate; spend here is for retention, not prevention |
| Low-Moderate Risk | ~20% | Maintain good behavior, prevent migration |
| Moderate-High Risk | ~40% | Largest segment, highest intervention ROI |
| High Risk | ~25% | Recovery-focused; each recovered account saves more per client |

The exact percentages should be calibrated to the bank's cost structure, which we do not have.

---

### 8.4 Sample campaign: "Payment Health Check," October 2026

**Objective:** Reduce default rate in the Moderate-High Risk segment through a 4-week multi-channel campaign timed to the year-end spending season, when payment stress typically increases.

**Target:** Moderate-High Risk clients with risk score above 45 (estimated 3,000-4,000 clients).

**Segment-specific tactics:**

| Segment | Offer | Email subject | App message |
|---|---|---|---|
| Low Risk | 2x reward points for holiday spending | "Earn double rewards this season" | "Your holiday rewards are ready" |
| Low-Mod Risk | Payment scheduling tool + on-time bonus | "Stay on track this holiday" | "Set up autopay and earn a bonus" |
| Mod-High Risk | Free financial counseling + payment restructuring | "Let's get your finances holiday-ready" | "Need help? Your advisor is one tap away" |
| High Risk | Emergency restructuring + hardship options | "We're here to help" | "Contact your specialist today" |

**Week-by-week:**
- Week 1: launch email campaign to all segments; set up landing pages; begin phone outreach to High Risk
- Week 2: follow-up SMS to non-responders in Mod-High and High Risk; start financial counseling sessions
- Week 3: phone calls to Mod-High Risk non-responders; send payment plan options
- Week 4: final urgency messaging; summarize results; schedule November follow-ups

**Expected results (with assumptions):**
- If the campaign prevents 5-10% of defaults in the Mod-High Risk segment, that is 110-220 fewer defaults
- At avg outstanding bill of NT$69K and 75% LGD, estimated savings: NT$5.7M to NT$11.4M
- Campaign costs depend on the bank's internal pricing for phone agents, messaging, and counseling, which we cannot estimate from this data

---

### 8.5 Implementation requirements

**Technology:** risk scoring engine (Python-based model deployment), CRM integration for scores and alerts, campaign automation platform for triggered multi-channel outreach, analytics dashboard for portfolio monitoring.

**People:** a credit risk analytics lead to manage model retraining and monitoring; 2 additional account specialists for proactive Moderate-High Risk outreach.

**Timeline:**
- Month 1-2: deploy scoring engine, integrate with CRM, configure alert rules, train staff
- Month 3-4: pilot campaign with 50% of Moderate-High Risk segment, measure results
- Month 5-6: roll out to full portfolio, implement credit limit policies
- Month 7-12: retrain model with new data, A/B test campaign variations, optimize budget allocation

We do not estimate costs for these items because they depend on the bank's existing infrastructure and internal pricing.

### 8.6 Expected financial impact

**What the data tells us:**
- 6,630 clients defaulted in the observed period
- Average outstanding bill for defaulters: NT$48,553
- Total exposure at default: NT$322M (6,630 x NT$48,553)

**Assumptions (with sources):**
- Loss Given Default: 75%. Basel framework prescribes 45% for senior unsecured claims; credit cards, being fully unsecured with lower recovery priority, typically see 60-80% in practice (Corporate Finance Institute; Wall Street Prep). We use 75%.
- Intervention effectiveness: 5-10% reduction in defaults. Based on a 2022 Journal of Banking & Finance study showing repayment reminders reduced delinquency progression by about 5%, and BAI Banking Strategies reporting that pre-delinquency management programs can prevent a portion of early-stage defaults.

**Estimates:**

| Metric | Value | Basis |
|---|---|---|
| Total exposure at default | NT$322M | Data: 6,630 x NT$48,553 avg bill |
| Estimated total loss (75% LGD) | NT$241M | Data + Basel LGD assumption |
| Loss per defaulter | NT$36,415 | NT$241M / 6,630 |
| If intervention prevents 5% of defaults | ~NT$12M saved | 332 fewer defaults x NT$36,415 |
| If intervention prevents 10% of defaults | ~NT$24M saved | 663 fewer defaults x NT$36,415 |

**What we cannot estimate:** implementation costs, technology investment, personnel costs, revenue impact of loyalty programs. Without these, we do not provide an ROI figure.


## 9. Limitations and future work

### 9.1 The actionability problem

The strongest predictors of default in our model are all measures of past delinquency: how many months the client has been late, how severe the worst delay was. This creates an inherent tension.

A client who is already 3 months behind will very likely miss the next payment too. But predicting that does not help much; the bank is already losing money. At the other end, for a brand new client with no payment history, the model has almost nothing to work with. The onboarding model (Section 5.4) confirms this: demographics and credit limit alone produce an AUC of only 0.605.

Where the model actually adds value is in the middle. Specifically:
1. The first missed payment, where risk jumps from 11.7% to 29.8%. This is the most actionable moment because the client has shown a behavioral change but is not yet deeply delinquent.
2. Pre-delinquency signals like rising utilization and declining payment ratios. These are weaker individually but can flag clients who are trending toward trouble before they miss a payment.
3. Portfolio-level resource allocation. Even imperfect individual predictions become useful when applied across 30,000 clients because the segmentation concentrates attention where it matters most.

### 9.2 Data limitations

- **Single time period:** the dataset covers April to September 2005 in Taiwan. We cannot validate whether these patterns hold across economic cycles or in other markets.
- **No income data:** income and employment status are arguably the most important determinants of repayment capacity, but they are not in the dataset.
- **No external credit data:** we do not know whether these clients have debts with other institutions.
- **Binary target:** "default" here means the client missed the next payment. It does not capture partial payments, severity, or duration.
- **Look-ahead in features:** engineered features like NUM_LATE and MAX_DELAY aggregate the full 6-month window. In a live deployment, these would be computed incrementally as each month's data arrives.

### 9.3 Methodological limitations

- SMOTE generates synthetic samples by interpolating between existing minority-class observations. If defaulters have distinct subgroups, SMOTE may create unrealistic hybrids.
- K-Means with k=4 was chosen over k=3 (which had a slightly higher silhouette score of 0.345 vs. 0.308) for business reasons. This tradeoff is defensible but subjective.
- Model performance plateaus around AUC 0.77, suggesting a natural ceiling given the available features. Additional data (income, transaction-level records, external credit scores) would likely push this higher.

### 9.4 Future directions

1. Build a dedicated new-client scorecard using application-time features plus external credit bureau data, since the current model cannot assess clients without payment history.
2. Reframe the prediction target: instead of "will they default next month," predict "will a currently good-standing client become delinquent within 6 months." This is more useful for prevention.
3. Run A/B tests on intervention strategies to measure actual causal impact rather than relying on observational correlations.
4. Incorporate macroeconomic variables (unemployment, interest rates) to make the model responsive to systemic risk shifts.


## 10. Conclusion

We analyzed 30,000 Taiwanese credit card clients to understand what drives default and how the bank can use that understanding operationally.

The core finding is that payment behavior, not demographics, determines default risk. Clients with zero late payments default at 11.7%. One late month pushes that to 29.8%. Six late months: 70.3%. Demographic features like age, education, and gender have correlations below |r| = 0.05 and produce an AUC of only 0.605 when used alone.

K-Means clustering identified four segments with default rates from 12.7% to 60.5%. The High Risk segment (12.4% of clients, 60.5% default rate) is clearly separated from the rest and accounts for a disproportionate share of losses. The Moderate-High Risk segment (38% of clients, 19.3% default rate) is the most promising target for intervention because it is large and not yet deeply delinquent.

The Random Forest model achieves AUC 0.770 with a top-decile lift of 3.07x, and a simple decision tree rule ("MAX_DELAY > 0 and PAY_0 >= 2") captures most of the predictive signal at AUC 0.761 without requiring any ML infrastructure.

Based on our data, the bank's total exposure at default is approximately NT$322M (6,630 defaulters x NT$48,553 average outstanding bill). Assuming a 75% loss-given-default rate, which is standard for unsecured credit card debt under Basel guidelines, estimated losses are around NT$241M. Published research (Journal of Banking & Finance, 2022) suggests early intervention programs can reduce delinquency by 5-10%, which would translate to roughly NT$12-24M in prevented losses for this portfolio. We cannot estimate implementation costs from this dataset, so we do not provide an ROI figure.

The main practical recommendations are: (1) deploy monthly risk scoring to prioritize attention, (2) implement the decision tree rule as an immediate alert system, and (3) target the Moderate-High Risk segment with outreach at the first sign of missed payments.


## 11. References

1. Yeh, I.-C., & Lien, C. (2009). The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients. *Expert Systems with Applications*, 36(2), 2473-2480.
2. UCI Machine Learning Repository. Default of Credit Card Clients Dataset. https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
3. Chawla, N. V., Bowyer, K. W., Hall, L. O., & Kegelmeyer, W. P. (2002). SMOTE: Synthetic Minority Over-sampling Technique. *Journal of Artificial Intelligence Research*, 16, 321-357.
4. Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *ACM SIGKDD*, 785-794.
5. Lundberg, S. M., & Lee, S.-I. (2017). A Unified Approach to Interpreting Model Predictions. *NeurIPS*, 30.
6. Loss Given Default, Corporate Finance Institute. https://corporatefinanceinstitute.com/resources/commercial-lending/loss-given-default-lgd/
7. Reducing credit card delinquency using repayment reminders, *Journal of Banking & Finance*, 2022. https://www.sciencedirect.com/science/article/abs/pii/S0378426622001431
8. Bolstering credit with pre-delinquency management, BAI Banking Strategies. https://www.bai.org/banking-strategies/bolstering-credit-with-pre-delinquency-management/

**Libraries:** Python 3.x, pandas, numpy, scikit-learn, XGBoost, SHAP, imbalanced-learn, matplotlib, seaborn
